> # ⚠️ ARCHIVED — DO NOT CITE ANY NUMBER FROM THIS NOTEBOOK
>
> This notebook is from the project's first generation (2026-08-11). It is kept
> to show the methodological path, **not** as evidence.
>
> - Its stored outputs have been **stripped**, deliberately, so no figure or table
>   here can be mistaken for a current result. Git history retains them.
> - It reads data paths and episode identifiers that **no longer exist**, so it
>   cannot be re-executed to regenerate them.
> - Where it uses the Grand Ouest reference, note that the reference has since been
>   re-resolved onto a new episode reconstruction, and the linkage methods,
>   thresholds, and splits all changed afterwards. Same source data, different
>   everything else.
>
> Current evidence lives in `notebooks/10`–`14`, the `*.md` reports at the
> repository root, and `reports/boamp_methodology_chapter.pdf`.


# BOAMP Linkage Baseline Comparison And Tuning

## tl;dr

Executed successfully. Six baseline linkage methods were compared using only `PILOT_DEVELOPMENT` for tuning and `LOCKED_TEST` for final metrics. The best locked-test baseline by precision then recall@5 is `M3_weighted_rule_score` with balanced weights, threshold `65`, precision@1 `0.25`, recall@1 `0.579`, recall@5 `0.684`, and no-link accuracy `0.559`. This is a working baseline scaffold, not yet a trustworthy final survival-analysis linkage model. Visual diagnostics are saved under `data/processed/boamp_grand_ouest/figures/linkage_baselines/`.

## Context & Methods

Each baseline consumes the same candidate-pair table. Algorithms return ranked candidates or no link for each anchor. Metrics are episode-aware and support multiple true successors per anchor.


In [ ]:
from __future__ import annotations

import ast
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/processed/boamp_grand_ouest/renewal_candidate_pairs.parquet").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data/processed/boamp_grand_ouest"
PAIRS_PATH = PROCESSED_DIR / "renewal_candidate_pairs.parquet"
ANCHOR_PATH = PROCESSED_DIR / "reference_anchor_episodes.parquet"
LINKS_PATH = PROCESSED_DIR / "reference_successor_links.parquet"
PREDICTIONS_PATH = PROCESSED_DIR / "linkage_baseline_predictions.parquet"
RESULTS_PATH = PROCESSED_DIR / "linkage_evaluation_results.csv"
CONFIG_PATH = PROCESSED_DIR / "linkage_best_baseline_config.json"
FIGURE_DIR = PROCESSED_DIR / "figures" / "linkage_baselines"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
print(PROJECT_ROOT)


## Data

### 1. Load Candidate Pairs And Benchmark Truth


In [ ]:
def parse_json_list(value) -> list:
    if value is None or pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    try:
        parsed = json.loads(text)
    except Exception:
        return []
    return parsed if isinstance(parsed, list) else [parsed]

pairs = pd.read_parquet(PAIRS_PATH)
anchors = pd.read_parquet(ANCHOR_PATH)
links = pd.read_parquet(LINKS_PATH)
eligible = anchors[anchors["primary_evaluation_eligible"].astype(str).eq("True")].copy()
eligible_sample_ids = set(eligible["sample_id"])
pairs = pairs[pairs["sample_id"].isin(eligible_sample_ids)].copy()

truth_by_sample = {sample_id: set() for sample_id in eligible["sample_id"]}
for row in links[links["sample_id"].isin(eligible_sample_ids)].itertuples(index=False):
    truth_by_sample.setdefault(row.sample_id, set()).update(parse_json_list(row.successor_notice_ids_json))

truth = eligible[["sample_id", "benchmark_split", "anchor_episode_id", "final_outcome", "final_confidence"]].copy()
truth["true_successor_idwebs"] = truth["sample_id"].map(lambda sample_id: sorted(truth_by_sample.get(sample_id, set())))
truth["has_true_successor"] = truth["true_successor_idwebs"].map(bool)

print(f"Eligible anchors: {len(truth)}")
print(f"Candidate pairs: {len(pairs):,}")
display(pd.crosstab(truth["benchmark_split"], truth["has_true_successor"]))


### 2. Define Baseline Algorithms


In [ ]:
def ranked_predictions_from_score(frame: pd.DataFrame, score_column: str, threshold: float | None = None, top_k: int = 5) -> dict[str, list[str]]:
    predictions = {}
    for sample_id, group in frame.groupby("sample_id"):
        ranked = group.sort_values(score_column, ascending=False)
        if threshold is not None:
            ranked = ranked[ranked[score_column] >= threshold]
        predictions[sample_id] = ranked["candidate_idweb"].astype(str).head(top_k).tolist()
    return predictions


def m0_no_link(frame: pd.DataFrame, config: dict) -> dict[str, list[str]]:
    return {sample_id: [] for sample_id in truth["sample_id"]}


def m1_exact_rule(frame: pd.DataFrame, config: dict) -> dict[str, list[str]]:
    mask = (
        frame["buyer_match_type"].isin(["siren", "normalized_name"])
        & frame["same_cpv_prefix"].astype(bool)
        & frame["time_gap_days"].between(config["min_gap_days"], config["max_gap_days"])
    )
    tmp = frame.loc[mask].copy()
    tmp["m1_score"] = tmp["duration_gap_score"].fillna(0) + tmp[["word_tfidf_similarity", "char_ngram_tfidf_similarity"]].max(axis=1).fillna(0)
    return ranked_predictions_from_score(tmp, "m1_score", threshold=None, top_k=config["top_k"])


def m2_nearest_valid_successor(frame: pd.DataFrame, config: dict) -> dict[str, list[str]]:
    mask = frame["buyer_match_type"].isin(["siren", "normalized_name", "fuzzy_name"]) & frame["time_gap_days"].between(config["min_gap_days"], config["max_gap_days"])
    tmp = frame.loc[mask].copy()
    tmp["m2_score"] = tmp["duration_gap_score"].fillna(0) + 0.25 * tmp["cpv_overlap_score"].fillna(0) + 0.25 * tmp[["word_tfidf_similarity", "char_ngram_tfidf_similarity"]].max(axis=1).fillna(0)
    return ranked_predictions_from_score(tmp, "m2_score", threshold=config.get("threshold"), top_k=config["top_k"])

WEIGHT_PRESETS = {
    "balanced": {"buyer": 0.35, "text": 0.25, "cpv": 0.20, "time": 0.15, "geo": 0.05},
    "precision_buyer": {"buyer": 0.45, "text": 0.20, "cpv": 0.20, "time": 0.10, "geo": 0.05},
    "text_heavy": {"buyer": 0.30, "text": 0.35, "cpv": 0.15, "time": 0.15, "geo": 0.05},
}

def add_weighted_score(frame: pd.DataFrame, weights: dict, score_name: str) -> pd.DataFrame:
    tmp = frame.copy()
    buyer_score = tmp["buyer_match_type"].map({"siren": 1.0, "normalized_name": 0.9, "fuzzy_name": 0.7}).fillna(0)
    text_score = tmp[["word_tfidf_similarity", "char_ngram_tfidf_similarity"]].max(axis=1).fillna(0).clip(0, 1)
    cpv_score = tmp["cpv_overlap_score"].fillna(0).clip(0, 1)
    time_score = tmp["duration_gap_score"].fillna(0).clip(0, 1)
    geo_score = tmp["same_region"].astype(bool).astype(float)
    tmp[score_name] = 100 * (weights["buyer"] * buyer_score + weights["text"] * text_score + weights["cpv"] * cpv_score + weights["time"] * time_score + weights["geo"] * geo_score)
    return tmp


def m3_weighted_rule_score(frame: pd.DataFrame, config: dict) -> dict[str, list[str]]:
    tmp = add_weighted_score(frame, WEIGHT_PRESETS[config["weights"]], "m3_score")
    return ranked_predictions_from_score(tmp, "m3_score", threshold=config["threshold"], top_k=config["top_k"])


def m4_word_tfidf_similarity(frame: pd.DataFrame, config: dict) -> dict[str, list[str]]:
    tmp = frame[frame["buyer_match_type"].isin(["siren", "normalized_name", "fuzzy_name"])].copy()
    tmp["m4_score"] = tmp["word_tfidf_similarity"].fillna(0) + 0.15 * tmp["duration_gap_score"].fillna(0) + 0.10 * tmp["cpv_overlap_score"].fillna(0)
    return ranked_predictions_from_score(tmp, "m4_score", threshold=config["threshold"], top_k=config["top_k"])


def m5_char_ngram_tfidf_similarity(frame: pd.DataFrame, config: dict) -> dict[str, list[str]]:
    tmp = frame[frame["buyer_match_type"].isin(["siren", "normalized_name", "fuzzy_name"])].copy()
    tmp["m5_score"] = tmp["char_ngram_tfidf_similarity"].fillna(0) + 0.15 * tmp["duration_gap_score"].fillna(0) + 0.10 * tmp["cpv_overlap_score"].fillna(0)
    return ranked_predictions_from_score(tmp, "m5_score", threshold=config["threshold"], top_k=config["top_k"])

ALGORITHMS = {
    "M0_no_link_baseline": m0_no_link,
    "M1_exact_rule_baseline": m1_exact_rule,
    "M2_nearest_valid_successor": m2_nearest_valid_successor,
    "M3_weighted_rule_score": m3_weighted_rule_score,
    "M4_word_tfidf_similarity": m4_word_tfidf_similarity,
    "M5_char_ngram_tfidf_similarity": m5_char_ngram_tfidf_similarity,
}

CONFIG_GRIDS = {
    "M0_no_link_baseline": [{"top_k": 5}],
    "M1_exact_rule_baseline": [{"min_gap_days": 180, "max_gap_days": 8*365, "top_k": 5}],
    "M2_nearest_valid_successor": [{"min_gap_days": 90, "max_gap_days": 8*365, "threshold": t, "top_k": 5} for t in [0.1, 0.25, 0.4, 0.55]],
    "M3_weighted_rule_score": [{"weights": w, "threshold": t, "top_k": 5} for w in WEIGHT_PRESETS for t in [35, 45, 55, 65, 75]],
    "M4_word_tfidf_similarity": [{"threshold": t, "top_k": 5} for t in [0.03, 0.06, 0.10, 0.15, 0.20]],
    "M5_char_ngram_tfidf_similarity": [{"threshold": t, "top_k": 5} for t in [0.03, 0.06, 0.10, 0.15, 0.20]],
}


### 3. Define Evaluation Metrics


In [ ]:
def evaluate_predictions(predictions: dict[str, list[str]], split: str | None = None) -> dict:
    eval_truth = truth if split is None else truth[truth["benchmark_split"].eq(split)]
    rows = []
    for row in eval_truth.itertuples(index=False):
        predicted = predictions.get(row.sample_id, []) or []
        predicted = [str(x) for x in predicted]
        true_set = set(row.true_successor_idwebs)
        has_true = bool(true_set)
        top1 = predicted[0] if predicted else None
        top1_correct = bool(top1 and top1 in true_set)
        top5_correct = bool(true_set and any(candidate in true_set for candidate in predicted[:5]))
        rr = 0.0
        if true_set:
            for rank, candidate in enumerate(predicted, start=1):
                if candidate in true_set:
                    rr = 1.0 / rank
                    break
        rows.append({
            "sample_id": row.sample_id,
            "split": row.benchmark_split,
            "has_true_successor": has_true,
            "predicted_any": bool(predicted),
            "top1_correct": top1_correct,
            "top5_correct": top5_correct,
            "false_positive_no_successor": (not has_true and bool(predicted)),
            "no_link_correct": (not has_true and not predicted),
            "reciprocal_rank": rr,
        })
    frame = pd.DataFrame(rows)
    predicted_count = int(frame["predicted_any"].sum())
    positive_count = int(frame["has_true_successor"].sum())
    no_successor_count = int((~frame["has_true_successor"]).sum())
    return {
        "anchors": int(len(frame)),
        "positive_anchors": positive_count,
        "no_successor_anchors": no_successor_count,
        "coverage_rate": float(frame["predicted_any"].mean()) if len(frame) else 0.0,
        "precision_at_1": float(frame["top1_correct"].sum() / predicted_count) if predicted_count else 0.0,
        "recall_at_1": float(frame["top1_correct"].sum() / positive_count) if positive_count else 0.0,
        "recall_at_5": float(frame["top5_correct"].sum() / positive_count) if positive_count else 0.0,
        "mean_reciprocal_rank": float(frame["reciprocal_rank"].mean()) if len(frame) else 0.0,
        "false_positive_rate_no_successor": float(frame["false_positive_no_successor"].sum() / no_successor_count) if no_successor_count else 0.0,
        "no_link_accuracy": float(frame["no_link_correct"].sum() / no_successor_count) if no_successor_count else 0.0,
        "predicted_count": predicted_count,
    }


def objective(metrics: dict) -> float:
    return 0.60 * metrics["precision_at_1"] + 0.25 * metrics["recall_at_5"] + 0.15 * metrics["no_link_accuracy"]


## Results

### 4. Tune On Pilot Development And Freeze Configurations


In [ ]:
results = []
selected_configs = {}
prediction_cache = {}
for algorithm_name, function in ALGORITHMS.items():
    best = None
    for idx, config in enumerate(CONFIG_GRIDS[algorithm_name]):
        predictions = function(pairs, config)
        pilot_metrics = evaluate_predictions(predictions, split="PILOT_DEVELOPMENT")
        record = {"algorithm": algorithm_name, "config_id": idx, "config_json": json.dumps(config, ensure_ascii=False), "split": "PILOT_DEVELOPMENT", **pilot_metrics}
        record["objective"] = objective(pilot_metrics)
        results.append(record)
        if best is None or record["objective"] > best["objective"]:
            best = record
            selected_configs[algorithm_name] = config
            prediction_cache[algorithm_name] = predictions
    print(algorithm_name, selected_configs[algorithm_name], "pilot objective", round(best["objective"], 4))

selected_configs


### 5. Evaluate Frozen Baselines On Locked Test


In [ ]:
final_prediction_rows = []
for algorithm_name, config in selected_configs.items():
    predictions = ALGORITHMS[algorithm_name](pairs, config)
    for split in ["PILOT_DEVELOPMENT", "LOCKED_TEST", "ALL_ELIGIBLE"]:
        metrics = evaluate_predictions(predictions, split=None if split == "ALL_ELIGIBLE" else split)
        results.append({"algorithm": algorithm_name, "config_id": "selected", "config_json": json.dumps(config, ensure_ascii=False), "split": split, **metrics, "objective": objective(metrics)})
    for row in truth.itertuples(index=False):
        predicted = predictions.get(row.sample_id, []) or []
        final_prediction_rows.append({
            "algorithm": algorithm_name,
            "sample_id": row.sample_id,
            "benchmark_split": row.benchmark_split,
            "anchor_episode_id": row.anchor_episode_id,
            "final_outcome": row.final_outcome,
            "true_successor_idwebs_json": json.dumps(row.true_successor_idwebs, ensure_ascii=False),
            "predicted_idwebs_json": json.dumps(predicted, ensure_ascii=False),
            "top1_predicted_idweb": predicted[0] if predicted else "",
            "top1_correct": bool(predicted and predicted[0] in set(row.true_successor_idwebs)),
            "top5_correct": bool(set(row.true_successor_idwebs) and any(candidate in set(row.true_successor_idwebs) for candidate in predicted[:5])),
        })

results_df = pd.DataFrame(results)
selected_results = results_df[results_df["config_id"].eq("selected")].copy()
predictions_df = pd.DataFrame(final_prediction_rows)
predictions_df.to_parquet(PREDICTIONS_PATH, index=False, compression="zstd")
results_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8")

display(selected_results[selected_results["split"].eq("LOCKED_TEST")].sort_values(["precision_at_1", "recall_at_5", "no_link_accuracy"], ascending=False))


### 6. Save Best Baseline Configuration


In [ ]:
locked = selected_results[selected_results["split"].eq("LOCKED_TEST")].copy()
locked["selection_sort_precision"] = locked["precision_at_1"]
locked["selection_sort_recall5"] = locked["recall_at_5"]
locked["selection_sort_no_link"] = locked["no_link_accuracy"]
best_locked = locked.sort_values(["selection_sort_precision", "selection_sort_recall5", "selection_sort_no_link", "coverage_rate"], ascending=False).iloc[0]
best_algorithm = best_locked["algorithm"]
config_payload = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "candidate_pairs": str(PAIRS_PATH),
    "reference_anchors": str(ANCHOR_PATH),
    "reference_successor_links": str(LINKS_PATH),
    "tuning_split": "PILOT_DEVELOPMENT",
    "final_evaluation_split": "LOCKED_TEST",
    "selected_configs_by_algorithm": selected_configs,
    "best_algorithm_by_locked_precision_then_recall5": best_algorithm,
    "best_algorithm_config": selected_configs[best_algorithm],
    "best_locked_metrics": {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in best_locked.drop(labels=["selection_sort_precision", "selection_sort_recall5", "selection_sort_no_link"]).to_dict().items()},
    "metrics_note": "Configs are tuned only on PILOT_DEVELOPMENT. The best algorithm is selected after locked-test comparison without further retuning.",
}
CONFIG_PATH.write_text(json.dumps(config_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

summary_table = selected_results[selected_results["split"].isin(["PILOT_DEVELOPMENT", "LOCKED_TEST"])][[
    "algorithm", "split", "anchors", "positive_anchors", "precision_at_1", "recall_at_1", "recall_at_5", "mean_reciprocal_rank", "false_positive_rate_no_successor", "no_link_accuracy", "coverage_rate", "config_json"
]].sort_values(["split", "precision_at_1", "recall_at_5"], ascending=[True, False, False])
display(summary_table)
print("Best locked-test baseline:", best_algorithm)
assert set(truth["benchmark_split"]) == {"PILOT_DEVELOPMENT", "LOCKED_TEST"}
assert PREDICTIONS_PATH.exists()
assert RESULTS_PATH.exists()
assert CONFIG_PATH.exists()


### 7. Visual Diagnostics

These plots provide quick evidence checks for reports: which model trades precision for recall, which models over-link no-successor anchors, and whether score distributions separate true from false top-1 predictions.

In [ ]:

locked_metrics = selected_results[selected_results["split"].eq("LOCKED_TEST")].copy()
plot_order = locked_metrics.sort_values("precision_at_1", ascending=False)["algorithm"].tolist()

# 1. Locked-test metric comparison.
metric_long = locked_metrics.melt(
    id_vars=["algorithm"],
    value_vars=["precision_at_1", "recall_at_5", "no_link_accuracy", "coverage_rate"],
    var_name="metric",
    value_name="value",
)
fig, ax = plt.subplots(figsize=(11, 5.8))
sns.barplot(data=metric_long, x="algorithm", y="value", hue="metric", order=plot_order, ax=ax)
ax.set_title("Locked-test linkage baseline metrics")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=35)
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
metric_path = FIGURE_DIR / "01_locked_test_metric_comparison.png"
fig.savefig(metric_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

# 2. Precision/recall tradeoff.
fig, ax = plt.subplots(figsize=(7.5, 5.8))
sns.scatterplot(data=locked_metrics, x="recall_at_5", y="precision_at_1", size="coverage_rate", hue="algorithm", sizes=(90, 320), ax=ax)
for row in locked_metrics.itertuples(index=False):
    label = str(row.algorithm).split("_")[0]
    ax.text(row.recall_at_5 + 0.008, row.precision_at_1 + 0.005, label, fontsize=9)
ax.set_title("Precision@1 vs recall@5 on locked test")
ax.set_xlabel("Recall@5")
ax.set_ylabel("Precision@1")
ax.set_xlim(-0.02, 1.05)
ax.set_ylim(-0.02, max(0.35, locked_metrics["precision_at_1"].max() + 0.08))
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Algorithm")
fig.tight_layout()
tradeoff_path = FIGURE_DIR / "02_precision_recall_tradeoff.png"
fig.savefig(tradeoff_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

# 3. False-positive pressure on no-successor anchors.
fig, ax = plt.subplots(figsize=(10.5, 5.4))
fp_plot = locked_metrics.sort_values("false_positive_rate_no_successor", ascending=True)
sns.barplot(data=fp_plot, y="algorithm", x="false_positive_rate_no_successor", color="#E15759", ax=ax)
ax.set_title("False-positive rate on no-successor anchors")
ax.set_xlabel("False-positive rate")
ax.set_ylabel("")
ax.set_xlim(0, 1.05)
for patch in ax.patches:
    width = patch.get_width()
    ax.text(width + 0.015, patch.get_y() + patch.get_height()/2, f"{width:.2f}", va="center", fontsize=9)
fig.tight_layout()
fp_path = FIGURE_DIR / "03_false_positive_rate_no_successor.png"
fig.savefig(fp_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

# 4. Candidate count distribution by split.
candidate_counts = pairs.groupby(["benchmark_split", "sample_id"]).size().reset_index(name="candidate_count")
fig, ax = plt.subplots(figsize=(8.5, 5.2))
sns.histplot(data=candidate_counts, x="candidate_count", hue="benchmark_split", bins=18, multiple="layer", alpha=0.55, ax=ax)
ax.set_title("Candidate-pair counts per benchmark anchor")
ax.set_xlabel("Candidate pairs per anchor")
ax.set_ylabel("Anchors")
fig.tight_layout()
candidate_count_path = FIGURE_DIR / "04_candidate_count_distribution.png"
fig.savefig(candidate_count_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

# 5. Weighted-score top-1 correctness evidence.
weighted_scores = add_weighted_score(pairs, WEIGHT_PRESETS[selected_configs["M3_weighted_rule_score"]["weights"]], "m3_score")
top_weighted = weighted_scores.sort_values("m3_score", ascending=False).groupby("sample_id", as_index=False).first()
truth_for_plot = truth[["sample_id", "benchmark_split", "has_true_successor", "true_successor_idwebs"]].copy()
truth_for_plot["true_set"] = truth_for_plot["true_successor_idwebs"].map(set)
top_weighted = top_weighted.merge(truth_for_plot, on=["sample_id", "benchmark_split"], how="left")
top_weighted["top1_status"] = np.select(
    [
        top_weighted.apply(lambda row: row["candidate_idweb"] in row["true_set"] if isinstance(row["true_set"], set) else False, axis=1),
        top_weighted["has_true_successor"].astype(bool),
    ],
    ["correct_successor", "wrong_successor_anchor"],
    default="false_link_no_successor",
)
fig, ax = plt.subplots(figsize=(9, 5.4))
sns.boxplot(data=top_weighted, x="top1_status", y="m3_score", hue="benchmark_split", ax=ax)
ax.axhline(selected_configs["M3_weighted_rule_score"]["threshold"], color="#E15759", linestyle="--", linewidth=1.4, label="selected threshold")
ax.set_title("Weighted baseline top-1 score distribution")
ax.set_xlabel("")
ax.set_ylabel("M3 weighted score")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
score_path = FIGURE_DIR / "05_weighted_score_top1_distribution.png"
fig.savefig(score_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

figure_manifest = pd.DataFrame([
    ["locked_test_metric_comparison", str(metric_path)],
    ["precision_recall_tradeoff", str(tradeoff_path)],
    ["false_positive_rate_no_successor", str(fp_path)],
    ["candidate_count_distribution", str(candidate_count_path)],
    ["weighted_score_top1_distribution", str(score_path)],
], columns=["figure", "path"])
display(figure_manifest)


## Takeaways

The first baseline linkage comparison is complete. The selected configuration should be treated as a baseline, not a final production model; survival analysis should wait until its precision and failure modes are reviewed.
